# 3B - Modeling su Dataset MACRO-BOND

## 3.1 - Import e Caricamento dati

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import f1_score, confusion_matrix, roc_auc_score, classification_report
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Caricamento del dataset
df_bond_macro_lagged = pd.read_csv('./data/df_bond_macro_lagged.csv', index_col=0 )

print(f"✓ Dataset preprocessato caricato: {df_bond_macro_lagged.shape[0]} righe e {df_bond_macro_lagged.shape[1]} colonne")
print(f"✓ Colonne disponibili: {df_bond_macro_lagged.columns.tolist()}")
print(f"NaN values per colonna:\n{df_bond_macro_lagged.isna().sum()}")

✓ Dataset preprocessato caricato: 176786 righe e 62 colonne
✓ Colonne disponibili: ['marketcode', 'referencedate', 'pricetype', 'pricevalue', 'volume', 'mintoday', 'maxtoday', 'description', 'redemptiondate', 'coupon', 'years_to_maturity', 'yield_to_maturity', 'fed_rate', 'vix', 'esi_index', 'euribor_3m', 'euribor_1y', 'hicp_euroarea', 'stoxx50', 'xeon', 'sega', 'interbank_stress_spread', 'bce_fed_spread', 'pricevalue_lag_7d', 'pricevalue_lag_15d', 'pricevalue_lag_30d', 'volume_lag_7d', 'volume_lag_15d', 'volume_lag_30d', 'fed_rate_lag_7d', 'fed_rate_lag_15d', 'fed_rate_lag_30d', 'vix_lag_7d', 'vix_lag_15d', 'vix_lag_30d', 'esi_index_lag_7d', 'esi_index_lag_15d', 'esi_index_lag_30d', 'euribor_3m_lag_7d', 'euribor_3m_lag_15d', 'euribor_3m_lag_30d', 'euribor_1y_lag_7d', 'euribor_1y_lag_15d', 'euribor_1y_lag_30d', 'hicp_euroarea_lag_7d', 'hicp_euroarea_lag_15d', 'hicp_euroarea_lag_30d', 'stoxx50_lag_7d', 'stoxx50_lag_15d', 'stoxx50_lag_30d', 'xeon_lag_7d', 'xeon_lag_15d', 'xeon_lag_30d', 

## 3.2 - Target Definition

A differenza dell'altro notebook questa volta proviamo a prevedere il prezzo del singolo bond e non della macroeconomia globale.


In [3]:
FORECAST_HORIZON = 30

# Ordiniamo per ISIN e Data per garantire la corretta sequenza temporale
df_bond_macro_lagged = df_bond_macro_lagged.sort_values(by=['isincode', 'referencedate']).copy()

#Creiamo il target sul Rendimento
df_bond_macro_lagged['ytm_target'] = df_bond_macro_lagged.groupby('isincode')['yield_to_maturity'].shift(-FORECAST_HORIZON)

# Scartiamo le righe con valori NaN nel target
df_bond_macro_lagged = df_bond_macro_lagged.dropna(subset=['ytm_target']).copy()

# Creiamo la variabile target binaria: 1 se il prezzo aumenta, 0 altrimenti
df_bond_macro_lagged['target'] = (df_bond_macro_lagged['ytm_target'] < df_bond_macro_lagged['yield_to_maturity']).astype(int)

print(f"\nTarget distribution:")
print(df_bond_macro_lagged['target'].value_counts())
print(f"Classe imbalance ratio: {(df_bond_macro_lagged['target'] == 1).sum() / (df_bond_macro_lagged['target'] == 0).sum():.3f}")


Target distribution:
target
0    87562
1    80588
Name: count, dtype: int64
Classe imbalance ratio: 0.920


## 3.3 - Modello Naive: Persistenza statica

In [4]:
# Isoliamo X and y
y_true = df_bond_macro_lagged['target']

y_naive_static = np.zeros_like(y_true)  # Prevediamo sempre la classe 0 (Euribor_3M non aumenterà)
print("--- Baseline 1: Static Persistence ---")
print(f"\nClassification Report:\n{classification_report(y_true, y_naive_static, zero_division=0)}") # zero_division=0 per evitare warning in caso di classi non predette

--- Baseline 1: Static Persistence ---

Classification Report:
              precision    recall  f1-score   support

           0       0.52      1.00      0.68     87562
           1       0.00      0.00      0.00     80588

    accuracy                           0.52    168150
   macro avg       0.26      0.50      0.34    168150
weighted avg       0.27      0.52      0.36    168150



Questi dati sono in linea con la distribuzione della classe di maggioranza. I modelli Più avanzati dovranno battere il **45% di accuratezza**.

## 3.4 - Modello Naive: Momentum Persistance

Valutiamo ora la persistenza del trend: predice che Euribor_3M aumenterà se è già in aumento negli ultimi 30 giorni.

In [5]:
# Assumiamo di usare la stessa classe del lag a 63 giorni come previsione (60 giorni di calendario = 63 giorni lavorativi)
# Scommetto 1 se il trend passato era in salita, 0 altrimenti

# trend_lag_30 = df_bond_macro_lagged['yield_to_maturity'] > df_bond_macro_lagged['yield_to_maturity_lag_30d']
# y_naive_trend = trend_lag_30.astype(int)

# print("--- Baseline 2: Lagged Trend ---")
# print(f"\nClassification Report:\n{classification_report(y_true, y_naive_trend, zero_division=0)}")
# print(f"ROC-AUC: {roc_auc_score(y_true, y_naive_trend):.3f}")

## 3.5 - Modello Random Forest

In [6]:
# elenco le colonne presenti nel dataset
df_bond_macro_lagged.columns

Index(['marketcode', 'referencedate', 'pricetype', 'pricevalue', 'volume',
       'mintoday', 'maxtoday', 'description', 'redemptiondate', 'coupon',
       'years_to_maturity', 'yield_to_maturity', 'fed_rate', 'vix',
       'esi_index', 'euribor_3m', 'euribor_1y', 'hicp_euroarea', 'stoxx50',
       'xeon', 'sega', 'interbank_stress_spread', 'bce_fed_spread',
       'pricevalue_lag_7d', 'pricevalue_lag_15d', 'pricevalue_lag_30d',
       'volume_lag_7d', 'volume_lag_15d', 'volume_lag_30d', 'fed_rate_lag_7d',
       'fed_rate_lag_15d', 'fed_rate_lag_30d', 'vix_lag_7d', 'vix_lag_15d',
       'vix_lag_30d', 'esi_index_lag_7d', 'esi_index_lag_15d',
       'esi_index_lag_30d', 'euribor_3m_lag_7d', 'euribor_3m_lag_15d',
       'euribor_3m_lag_30d', 'euribor_1y_lag_7d', 'euribor_1y_lag_15d',
       'euribor_1y_lag_30d', 'hicp_euroarea_lag_7d', 'hicp_euroarea_lag_15d',
       'hicp_euroarea_lag_30d', 'stoxx50_lag_7d', 'stoxx50_lag_15d',
       'stoxx50_lag_30d', 'xeon_lag_7d', 'xeon_lag_15d', 'x

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, roc_auc_score
import numpy as np
import joblib

# Prepariamo i dati
df_bond_macro_lagged = df_bond_macro_lagged.sort_values(by=['referencedate']).reset_index(drop=True)

columns_to_exclude = ['marketcode', 'referencedate', 'pricetype', 'ytm_target', 'target', 'redemptiondate', 'description', 'isincode']
feature_columns = [col for col in df_bond_macro_lagged.columns if col not in columns_to_exclude]

X = df_bond_macro_lagged[feature_columns].to_numpy(dtype=float)
y = df_bond_macro_lagged['target'].to_numpy(dtype=int)

# Creiamo gli indici personalizzati per la Cross-Validation
unique_dates = np.sort(df_bond_macro_lagged['referencedate'].unique())
n_splits = 5
test_size_days = len(unique_dates) // (n_splits + 1)
gap_days = FORECAST_HORIZON

# Questa lista conterrà le tuple (indici_train, indici_test) per ogni fold
custom_cv_splits = []

for fold in range(n_splits):
    test_start_idx = len(unique_dates) - test_size_days * (n_splits - fold)
    test_end_idx = test_start_idx + test_size_days
    train_end_idx = test_start_idx - gap_days

    train_dates = unique_dates[:train_end_idx]
    test_dates = unique_dates[test_start_idx:test_end_idx]

    # Troviamo la posizione (indice numerico) delle righe che corrispondono a quelle date
    train_indices = np.where(df_bond_macro_lagged['referencedate'].isin(train_dates))[0]
    test_indices = np.where(df_bond_macro_lagged['referencedate'].isin(test_dates))[0]

    custom_cv_splits.append((train_indices, test_indices))

# CREIAMO LA PIPELINE E LO SPAZIO DI RICERCA
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('rf', RandomForestClassifier(random_state=42))
])

param_dist = {
    'rf__n_estimators': [50, 100, 200],
    'rf__max_depth': [3, 5, 7, 10],
    'rf__min_samples_split': [2, 5, 10]
}

print("Avvio Ottimizzazione Iperparametri con Custom Time Split...")

# RANDOMIZED SEARCH CV
random_search = RandomizedSearchCV(
    estimator=pipe,
    param_distributions=param_dist,
    n_iter=5,                   # Numero di combinazioni da provare (aumenta se hai tempo)
    scoring='roc_auc',
    cv=custom_cv_splits,
    n_jobs=-1,
    random_state=42,
    verbose=1
)

random_search.fit(X, y)

print("\n" + "="*50)
print(f"Miglior ROC-AUC medio ottenuto: {random_search.best_score_:.3f}")
print("Migliori Iperparametri trovati:")
for param, value in random_search.best_params_.items():
    print(f" - {param.replace('rf__', '')}: {value}")

# Salviamo il modello vincitore per SHAP
joblib.dump(random_search.best_estimator_, './data/best_rf_bonds_macro_model.pkl')
print("✓ Modello Random Forest salvato con successo!")

Avvio Ottimizzazione Iperparametri con Custom Time Split...
Fitting 5 folds for each of 5 candidates, totalling 25 fits

Miglior ROC-AUC medio ottenuto: 0.583
Migliori Iperparametri trovati:
 - n_estimators: 200
 - min_samples_split: 10
 - max_depth: 7
✓ Modello Random Forest salvato con successo!


## 3.6 - Modello LSTM (Long Short-Term Memory)

In [8]:
def create_sequences(X_data, y_data, seq_length):
    """
    Trasforma array 2D in array 3D [samples, seq_length, features].
    """
    xs, ys = [], []
    for i in range(len(X_data) - seq_length + 1): # Aggiunto +1 per non perdere l'ultima riga
        # Prende la finestra di giorni (es. da 0 a 59)
        xs.append(X_data[i : (i + seq_length)])

        # Il target DEVE essere quello associato all'ULTIMO giorno della finestra
        # L'ultimo giorno della finestra è (i + seq_length - 1)
        ys.append(y_data[i + seq_length - 1])

    return np.array(xs), np.array(ys)

In [9]:
# Creiamo la classe LSTM
class MacroLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout_rate):
        super(MacroLSTM, self).__init__()

        # Se c'è 1 solo layer, diciamo a PyTorch che il dropout interno è 0.
        lstm_dropout = dropout_rate if num_layers > 1 else 0.0

        # layers LSTM
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=lstm_dropout) #  batch_first=True è FONDAMENTALE perché i nostri dati avranno forma (batch_size, seq_length, features)

        self.dropout = nn.Dropout(dropout_rate) # dropout per regolarizzazione

        # layer finale di classificazione
        self.fc = nn.Linear(hidden_size, 1)  # output binario

    def forward(self, x):
        lstm_out, (hn, cn) = self.lstm(x)
        last_time_step_out = lstm_out[:, -1, :]
        out = self.dropout(last_time_step_out)
        out = self.fc(out)
        return out

In [ ]:
from sklearn.preprocessing import StandardScaler

# Importiamo il dataset macro senza lag
df_bond_macro_pp = pd.read_csv('./data/df_bond_macro.csv', index_col=0).reset_index()

# Assicuriamoci che i nomi siano corretti (se reset_index la chiama 'index', la rinominiamo)
if 'index' in df_bond_macro_pp.columns:
    df_bond_macro_pp = df_bond_macro_pp.rename(columns={'index': 'isincode'})
    print("✓ Colonna 'index' rinominata in 'isincode'")

# Definiamo la lunghezza dell'orizzonte di previsione
SEQ_LENGTH = 45

# Assicuriamoci che le date siano datetime e ordiniamo temporalmente l'intero dataset
df_bond_macro_pp['referencedate'] = pd.to_datetime(df_bond_macro_pp['referencedate'])
df_bond_macro_pp = df_bond_macro_pp.sort_values(['referencedate', 'isincode'])

# Creiamo la colonna col valore futuro
df_bond_macro_pp['YTM_target'] = df_bond_macro_pp.groupby('isincode')['yield_to_maturity'].shift(-FORECAST_HORIZON)
df_bond_macro_pp = df_bond_macro_pp.dropna(subset=['YTM_target']).copy()
df_bond_macro_pp['target'] = (df_bond_macro_pp['YTM_target'] > df_bond_macro_pp['yield_to_maturity']).astype(int)

# Droppiamo i valori NaN creati dallo shift
df_bond_macro_pp = df_bond_macro_pp.dropna(subset=['YTM_target']).copy()

# Selezioniamo le feature base
feature_cols = ['pricevalue', 'volume',
       'mintoday', 'maxtoday', 'coupon',
       'years_to_maturity', 'pricevalue',
       'fed_rate', 'vix', 'esi_index',
       'euribor_3m', 'euribor_1y', 'hicp_euroarea', 'stoxx50', 'xeon', 'sega',
       'interbank_stress_spread', 'bce_fed_spread']

unique_dates = np.sort(df_bond_macro_pp['referencedate'].unique())
n_splits = 5
test_size_days = len(unique_dates) // (n_splits + 2)
gap_days = SEQ_LENGTH

lstm_roc_auc_scores = []
lstm_f1_scores = []

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

precomputed_folds = []

for fold in range(n_splits):
       print(f"\n--- Fold {fold+1} ---")

       # Identifichiamo la data esatta in cui deve INIZIARE la valutazione del test
       eval_start_idx = len(unique_dates) - test_size_days * (n_splits - fold)
       eval_end_idx = eval_start_idx + test_size_days

       # Le date del TEST includono la finestra di lookback passata per non rompere le sequenze
       test_start_idx = eval_start_idx - (SEQ_LENGTH - 1)

       train_dates = unique_dates[:eval_start_idx - gap_days]
       test_dates = unique_dates[test_start_idx:eval_end_idx]

       train_mask = df_bond_macro_pp['referencedate'].isin(train_dates)
       test_mask = df_bond_macro_pp['referencedate'].isin(test_dates)

       # SPLIT: Dividiamo il DataFrame mantenendo tutte le colonne
       df_train = df_bond_macro_pp[train_mask].copy()
       df_test = df_bond_macro_pp[test_mask].copy()

       # SCALING: Facciamo fit solo sul train
       scaler = StandardScaler()
       df_train[feature_cols] = scaler.fit_transform(df_train[feature_cols])
       df_test[feature_cols] = scaler.transform(df_test[feature_cols])

       # CREAZIONE SEQUENZE: Per ogni ISIN creiamo le sequenze di input per LSTM
       X_train_seq_list, y_train_seq_list = [], []
       X_test_seq_list, y_test_seq_list = [], []

       # Ciclo per train
       for isin in df_train['isincode'].unique():
              df_isin = df_train[df_train['isincode'] == isin].sort_values(by='referencedate')

              # creiamo sequenze solo se ci sono abbastanza giorni
              if len(df_isin) > SEQ_LENGTH:
                     X_seq, y_seq = create_sequences(df_isin[feature_cols].values, df_isin['target'].values, SEQ_LENGTH)
                     X_train_seq_list.append(X_seq)
                     y_train_seq_list.append(y_seq)

       # Ciclo per Test
       for isin in df_test['isincode'].unique():
              df_isin = df_test[df_test['isincode'] == isin].sort_values(by='referencedate')

              # creiamo sequenze solo se ci sono abbastanza giorni
              if len(df_isin) >= SEQ_LENGTH:
                     X_seq, y_seq = create_sequences(df_isin[feature_cols].values, df_isin['target'].values, SEQ_LENGTH)

                     # df_isin ha capienza maggiorata, le prime (SEQ_lenght-1) sequenze appartengono al passato
                     # le scartiamo per valutare soltanto i giorni corretti di test
                     valid_test_sequences = len(df_isin) - (SEQ_LENGTH - 1)
                     X_seq = X_seq[-valid_test_sequences:]
                     y_seq = y_seq[-valid_test_sequences:]

                     X_test_seq_list.append(X_seq)
                     y_test_seq_list.append(y_seq)


       # --- SAFETY CHECK ---
       if not X_train_seq_list or not X_test_seq_list:
           print("ERRORE: Non ci sono abbastanza giorni per creare le sequenze in questo Fold. Salto...")
           continue

       # Concateniamo tutte le sequenze
       X_train_3d = np.concatenate(X_train_seq_list)
       y_train_seq = np.concatenate(y_train_seq_list)
       X_test_3d = np.concatenate(X_test_seq_list)
       y_test_seq = np.concatenate(y_test_seq_list)

       # Convertiamo in tensori PyTorch
       X_train_tensor = torch.tensor(X_train_3d, dtype=torch.float32)
       y_train_tensor = torch.tensor(y_train_seq, dtype=torch.float32).unsqueeze(1) # Aggiungiamo dimensione per BCE Loss
       X_test_tensor = torch.tensor(X_test_3d, dtype=torch.float32)
       y_test_tensor = torch.tensor(y_test_seq, dtype=torch.float32).unsqueeze(1) # Aggiungiamo dimensione per BCE Loss

       # DataLoaders (per addestrare a "pacchetti" e non saturare la RAM)
       train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=32, shuffle=False)
       test_loader = DataLoader(TensorDataset(X_test_tensor, y_test_tensor), batch_size=32, shuffle=False)

       input_dim = X_train_3d.shape[2]
       precomputed_folds.append({
              'train_loader': train_loader,
              'test_loader': test_loader,
              'input_dim': input_dim
       })

print("✓ Tutti i fold sono stati processati e salvati in memoria!")

Using device: cuda

--- Fold 1 ---

--- Fold 2 ---

--- Fold 3 ---

--- Fold 4 ---

--- Fold 5 ---
✓ Tutti i fold sono stati processati e salvati in memoria!


### Funzione per Optuna
Creiamo la funzione che Optuna chiamerà ripetutamente. Questa funzione prenderà i dati salvati e cercherà l'architettura perfetta.

In [12]:
import optuna

def objective(trial):
    # 1. Chiediamo a Optuna di "suggerire" i parametri per questo tentativo
    hidden_size = trial.suggest_categorical('hidden_size', [16, 32, 64])
    num_layers = trial.suggest_int('num_layers', 1, 3)
    dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.5)
    lr = trial.suggest_float('lr', 1e-5, 1e-2, log=True)
    epochs = trial.suggest_int('epochs', 5, 20)

    fold_roc_aucs = []

    # 2. Addestriamo il modello suggerito sui nostri 5 fold pre-calcolati
    for fold_data in precomputed_folds:
        model = MacroLSTM(
            input_size=fold_data['input_dim'],
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout_rate=dropout_rate
        ).to(device)

        criterion = nn.BCEWithLogitsLoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)

        # Training
        for epoch in range(epochs):
            model.train()
            for batch_X, batch_y in fold_data['train_loader']:
                batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                optimizer.zero_grad()
                loss = criterion(model(batch_X), batch_y)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0) # Protezione gradienti
                optimizer.step()

        # Evaluation
        model.eval()
        y_preds_prob = []
        y_trues = []
        with torch.no_grad():
            for batch_X, batch_y in fold_data['test_loader']:
                batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                probs = torch.sigmoid(model(batch_X))
                y_preds_prob.extend(probs.cpu().numpy().flatten())
                y_trues.extend(batch_y.cpu().numpy().flatten())

        # Calcoliamo il ROC-AUC del fold e lo salviamo
        roc = roc_auc_score(y_trues, y_preds_prob)
        fold_roc_aucs.append(roc)

    # 3. La funzione DEVE restituire il valore che vogliamo massimizzare
    return np.mean(fold_roc_aucs)

In [13]:
print("Avvio Ottimizzazione Iperparametri LSTM con Optuna...")

# Creiamo lo "Studio" dicendogli che vogliamo massimizzare il ROC-AUC
study = optuna.create_study(direction='maximize')

# Facciamo 15 tentativi
study.optimize(objective, n_trials=15)

print("\n" + "="*50)
print(f"Miglior ROC-AUC medio ottenuto: {study.best_value:.3f}")
print("Migliori Iperparametri trovati:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

[I 2026-07-06 13:39:22,881] A new study created in memory with name: no-name-6b41e0e7-23ae-4441-8723-6bd43dcc1405


Avvio Ottimizzazione Iperparametri LSTM con Optuna...


[I 2026-07-06 13:52:12,040] Trial 0 finished with value: 0.4853669835800935 and parameters: {'hidden_size': 16, 'num_layers': 2, 'dropout_rate': 0.4440920376569891, 'lr': 0.00027183894141558323, 'epochs': 19}. Best is trial 0 with value: 0.4853669835800935.
[I 2026-07-06 14:00:10,680] Trial 1 finished with value: 0.4525699147328461 and parameters: {'hidden_size': 64, 'num_layers': 2, 'dropout_rate': 0.49596193107682673, 'lr': 1.7805907406550008e-05, 'epochs': 12}. Best is trial 0 with value: 0.4853669835800935.
[I 2026-07-06 14:09:50,791] Trial 2 finished with value: 0.5103741961051863 and parameters: {'hidden_size': 16, 'num_layers': 3, 'dropout_rate': 0.28166491696867935, 'lr': 5.508426469458716e-05, 'epochs': 13}. Best is trial 2 with value: 0.5103741961051863.
[I 2026-07-06 14:12:39,610] Trial 3 finished with value: 0.5891962630774934 and parameters: {'hidden_size': 64, 'num_layers': 1, 'dropout_rate': 0.4191786614233358, 'lr': 0.007421464721721697, 'epochs': 5}. Best is trial 3 wi


Miglior ROC-AUC medio ottenuto: 0.626
Migliori Iperparametri trovati:
  hidden_size: 32
  num_layers: 1
  dropout_rate: 0.22033494894127748
  lr: 0.001933052440773484
  epochs: 5


### 3.6.1 - Addestramento del Modello LSTM Definitivo e Salvataggio
Utilizziamo i migliori iperparametri trovati da Optuna per calcolare le metriche
complete (incluso F1-Score con soglia rigorosa a 0.5 per evitare data leakage)
 e salviamo il modello.

In [14]:
print("\n" + "="*50)
print("ADDESTRAMENTO MODELLO LSTM FINALE")
print("="*50)

# 1. Estraiamo i parametri vincenti da Optuna
best_hidden_size = study.best_params['hidden_size']
best_num_layers = study.best_params['num_layers']
best_dropout = study.best_params['dropout_rate']
best_epochs = study.best_params['epochs']
best_lr = study.best_params['lr']

final_roc_aucs = []
final_f1_scores = []

for fold, fold_data in enumerate(precomputed_folds):
    print(f"\n--- Training Final LSTM - Fold {fold+1} ---")

    model = MacroLSTM(
        input_size=fold_data['input_dim'],
        hidden_size=best_hidden_size,
        num_layers=best_num_layers,
        dropout_rate=best_dropout
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=best_lr, weight_decay=1e-4) # Aggiunto weight_decay per stabilizzare
    criterion = nn.BCEWithLogitsLoss()

    # --- TRAINING ---
    for epoch in range(best_epochs):
        model.train()
        for batch_X, batch_y in fold_data['train_loader']:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(batch_X), batch_y)
            loss.backward()
            optimizer.step()

    # CALIBRAZIONE DELLA SOGLIA SUL TRAIN SET

    model.eval()
    train_probs = []
    train_trues = []
    with torch.no_grad():
        for batch_X, batch_y in fold_data['train_loader']:
            batch_X = batch_X.to(device)
            probs = torch.sigmoid(model(batch_X))
            train_probs.extend(probs.cpu().numpy().flatten())
            train_trues.extend(batch_y.numpy().flatten())

    # --- EVALUATION SUL TEST SET DEL FOLD ---
    y_preds_prob = []
    y_trues = []

    with torch.no_grad():
        for batch_X, batch_y in fold_data['test_loader']:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            probs = torch.sigmoid(model(batch_X))
            y_preds_prob.extend(probs.cpu().numpy().flatten())
            y_trues.extend(batch_y.cpu().numpy().flatten())

    # 3. Applichiamo la SOGLIA OTTIMALE (imparata dal passato) sui dati futuri
    y_preds_class = [1 if p > 0.5 else 0 for p in y_preds_prob]

    # Calcolo metriche
    roc = roc_auc_score(y_trues, y_preds_prob)
    f1 = f1_score(y_trues, y_preds_class)


    final_roc_aucs.append(roc)
    final_f1_scores.append(f1)

    print(f"Fold ROC-AUC: {roc:.3f} | F1: {f1:.3f} ")
    print(f"Confusion Matrix:\n{confusion_matrix(y_trues, y_preds_class)}")

# Risultati Finali
print("\n" + "="*50)
print(f"RISULTATO FINALE LSTM OPTIMIZED (Media su {n_splits} folds):")
print(f"Mean ROC-AUC : {np.mean(final_roc_aucs):.3f}")
print(f"Mean F1-Score: {np.mean(final_f1_scores):.3f}")
print("="*50)


ADDESTRAMENTO MODELLO LSTM FINALE

--- Training Final LSTM - Fold 1 ---
Fold ROC-AUC: 0.565 | F1: 0.827 
Confusion Matrix:
[[    0  6584]
 [    0 15705]]

--- Training Final LSTM - Fold 2 ---
Fold ROC-AUC: 0.557 | F1: 0.405 
Confusion Matrix:
[[  828 17922]
 [   73  6121]]

--- Training Final LSTM - Fold 3 ---
Fold ROC-AUC: 0.814 | F1: 0.303 
Confusion Matrix:
[[14125   365]
 [10365  2333]]

--- Training Final LSTM - Fold 4 ---
Fold ROC-AUC: 0.588 | F1: 0.047 
Confusion Matrix:
[[13314   268]
 [14912   377]]

--- Training Final LSTM - Fold 5 ---
Fold ROC-AUC: 0.563 | F1: 0.610 
Confusion Matrix:
[[ 5389  3642]
 [ 9384 10169]]

RISULTATO FINALE LSTM OPTIMIZED (Media su 5 folds):
Mean ROC-AUC : 0.618
Mean F1-Score: 0.438
